IMPORTING MODULES

In [ ]:
import pandas as pd
import mne
import numpy as np
from mne.time_frequency import psd_array_welch
from antropy import hjorth_params
from antropy.entropy import spectral_entropy, perm_entropy
import scipy.stats as stats
import os
from glob import glob
import torch
import torch.nn as nn
import torch.optim as optim
import math
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from torch.utils.data import TensorDataset, DataLoader
import kagglehub

LABEL MAP

In [ ]:
label_map = {
    "Relax": 0,
    "Stroop": 1,
    "Arithmetic": 1,
    "Mirror_image": 1
}

PREPROCESSING AND FEATURE EXTRACTION

In [ ]:
def preprocess_trial(trial, sfreq):
    n_channels = trial.shape[0]
    ch_names = [f'ch{i+1}' for i in range(32)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=['eeg'] * n_channels, verbose=False)
    raw = mne.io.RawArray(trial, info, verbose=False)

    raw.filter(1., 60., verbose=False)
    raw.notch_filter(freqs=[50], verbose=False)
    raw.set_eeg_reference('average', verbose=False)

    return raw

In [ ]:
freq_bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45)
}

In [ ]:
def extract_features_from_trial(raw, sf):
    features = []
    for ch_data in raw.get_data():
        features.append(np.mean(ch_data))
        features.append(np.std(ch_data))
        features.append(stats.skew(ch_data))
        features.append(stats.kurtosis(ch_data))

        hjorth = hjorth_params(ch_data)
        features.extend(hjorth)

        psd, freqs = psd_array_welch(ch_data, sf, fmin=0.5, fmax=45, n_fft=256, verbose=False)
        total_power = np.sum(psd)

        for band, (fmin, fmax) in freq_bands.items():
            band_power = np.sum(psd[(freqs >= fmin) & (freqs < fmax)])
            features.append(band_power / total_power if total_power > 0 else 0)

        features.append(spectral_entropy(ch_data, sf=sf, method="welch", normalize=True))
        features.append(perm_entropy(ch_data, order=3, normalize=True))

    return features

DATASET

In [ ]:
path = kagglehub.dataset_download("ayushtibrewal/raw-eeg-stress-dataset-sam40")
root = path

In [ ]:
activities = ["Relax", "Stroop", "Arithmetic", "Mirror_image"]
sfreq = 128

data = {}
X, y = [], []

In [ ]:
for activity in activities:
    activity_folders = [f for f in os.listdir(root) if f.startswith(activity)]
    if not activity_folders:
        print(f"No folder found for {activity}")
        continue
    activity_path = os.path.join(root, activity_folders[0], activity)
    csv_files = sorted(glob(os.path.join(activity_path, "*.csv")))

    for f in csv_files:
        subject_id = os.path.basename(f).split("_")[-1].replace(".csv", "")
        if subject_id not in data:
            data[subject_id] = {}

        df = pd.read_csv(f)
        data[subject_id][activity] = df

        trial = df.values[:, 1:].T
        raw = preprocess_trial(trial, sfreq=sfreq)
        features = extract_features_from_trial(raw, sfreq)

        X.append(features)
        y.append(label_map[activity])


In [ ]:
X = np.array(X)
X = X.reshape(160, 13, 32)
X_transformer = X.transpose(0, 2, 1)

TRANSFORMER MODEL

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, hidden, drop_prob=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, hidden)
        self.linear2 = nn.Linear(hidden, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=drop_prob)
    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [ ]:
class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
    def forward(self, q, k, v, mask=None):
        _, _, _, d_tensor = k.size()
        k_t = k.transpose(2, 3)
        score = (q @ k_t) / math.sqrt(d_tensor)
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e9)
        score = self.softmax(score)
        return score @ v, score


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.attention = ScaleDotProductAttention()
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_concat = nn.Linear(d_model, d_model)
    def forward(self, q, k, v, mask=None):
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)
        q, k, v = self.split(q), self.split(k), self.split(v)
        out, _ = self.attention(q, k, v, mask=mask)
        return self.w_concat(self.concat(out))
    def split(self, tensor):
        batch_size, length, d_model = tensor.size()
        d_tensor = d_model // self.n_head
        return tensor.view(batch_size, length, self.n_head, d_tensor).transpose(1, 2)
    def concat(self, tensor):
        batch_size, head, length, d_tensor = tensor.size()
        return tensor.transpose(1, 2).contiguous().view(batch_size, length, head * d_tensor)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-12):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, unbiased=False, keepdim=True)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, hidden_dim, drop_prob=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_head)
        self.norm1 = LayerNorm(d_model)
        self.ffn = PositionwiseFeedForward(d_model, hidden_dim, drop_prob)
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(drop_prob)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attention(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_head, hidden_dim, num_layers, drop_prob=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_head, hidden_dim, drop_prob)
            for _ in range(num_layers)
        ])
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [ ]:
class EEGTransformer(nn.Module):
    def __init__(self, feature_dim=13, d_model=64, n_head=4, hidden_dim=128, num_layers=2, drop_prob=0.2, num_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.encoder = TransformerEncoder(d_model, n_head, hidden_dim, num_layers, drop_prob)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, mask=None, predict=False):
        x = self.input_proj(x)
        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.encoder(x, mask)
        cls_out = x[:, 0]
        logits = self.classifier(cls_out)
        if predict:
            preds = torch.argmax(logits, dim=1)
            return preds.unsqueeze(1)
        return logits

NORMALIZATION

In [ ]:
X_mean = X_transformer.mean(axis=(0, 1), keepdims=True)
X_std = X_transformer.std(axis=(0, 1), keepdims=True) + 1e-8
X_transformer = (X_transformer - X_mean) / X_std

VARIABLES

In [ ]:
model = EEGTransformer()
class_weights = torch.tensor([1.0, 2.0])
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

X_tensor = torch.tensor(X_transformer, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

n_samples, n_channels, n_features = X_transformer.shape
X_flat = X_transformer.reshape((n_samples, -1))

smote = SMOTE(sampling_strategy='minority', random_state=42)
X_res, y_res = smote.fit_resample(X_flat, y)
X_res = X_res.reshape((-1, n_channels, n_features))
X_res_tensor = torch.tensor(X_res, dtype=torch.float32)
y_res_tensor = torch.tensor(y_res, dtype=torch.long)




LOADING AND SPLITING DATA

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_res_tensor, y_res_tensor, test_size=0.25, random_state=42, stratify=y_res
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

TRAINING

In [ ]:
for epoch in range(50):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()

    avg_loss = total_loss / len(train_loader.dataset)
    acc = correct / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Train Accuracy: {acc:.4f}")
    scheduler.step(avg_loss)

EVALUATION

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()

test_acc = correct / len(test_loader.dataset)
print(f"Test Accuracy: {test_acc:.4f}")

preds = model(X_tensor, predict=True)


CLASSIFICATION REPORT

In [ ]:
y_true, y_pred = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y_batch.tolist())
        y_pred.extend(preds.tolist())

print(classification_report(y_true, y_pred, digits=4))